# Matter and Thread: version progression and the standardisation debate

**Claim cluster:** `C001`, `C002`, `C003`, `C004`, `C005`, `C006`, `C009`, `C011`  
**Source of truth:** `data/smarthome.duckdb` (read-only), via `src/viz_core.py`

In [ ]:
import os
import sys
from pathlib import Path

import ipywidgets as widgets
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import HTML, clear_output, display

# Find the repo root by walking up until schemas/stage2.sql appears, so the
# notebook runs from notebooks/, from the repo root, or from anywhere else.
_here = Path.cwd().resolve()
for _candidate in [_here, *_here.parents]:
    if (_candidate / "schemas" / "stage2.sql").exists():
        ROOT = _candidate
        break
else:  # pragma: no cover - only reachable outside a checkout
    raise FileNotFoundError("schemas/stage2.sql not found above " + str(_here))

if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

# Option A topology: this wave reads one dedicated Stage 2 database. Point
# viz_core at it before import -- it resolves the path once at import time --
# so every fetch in this kernel uses the same file.
os.environ["SMARTHOME_DB"] = str(ROOT / "data" / "smarthome.duckdb")

import viz_core as viz  # noqa: E402

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 160)

# Renderers matter for two reasons. "browser" (plotly's default) opens a
# window when a figure is shown outside a live kernel, which breaks headless
# execution. The bare "notebook" renderer in plotly 7 emits only text/html,
# carrying no figure payload, so a notebook could run clean and still show
# nothing. "plotly_mimetype+notebook" emits the figure JSON for JupyterLab and
# an inline HTML fallback elsewhere -- which is also what the headless test
# asserts on.
pio.renderers.default = "plotly_mimetype+notebook"

# Every figure in this notebook reads only from data/smarthome.duckdb (read-only).
CLAIM_IDS = ('C001', 'C002', 'C003', 'C004', 'C005', 'C006', 'C009', 'C011')
assert viz.DEFAULT_DB_PATH.endswith("smarthome.duckdb"), viz.DEFAULT_DB_PATH

claims = viz.fetch_claims(list(CLAIM_IDS))
sources = viz.sources_for_claims(list(CLAIM_IDS)) if CLAIM_IDS else pd.DataFrame()
cluster_metrics = (
    viz.claim_metrics_lookup(list(CLAIM_IDS)) if CLAIM_IDS else pd.DataFrame()
)

# Corpus-wide row counts, read once so a notebook can state the size of what it
# is drawing from without confusing a whole-corpus count with the size of its
# own (possibly empty) claim cluster.
CORPUS = viz.connect_db()
try:
    corpus_counts = {
        table: CORPUS.execute(f"SELECT COUNT(1) FROM {table}").fetchone()[0]
        for table in ("stage2_claim", "metric", "entity", "source_mirror",
                      "predicate", "extraction_decision")
    }
finally:
    CORPUS.close()

def provenance(claim_id):
    """Collapsible evidence panel for one claim in this cluster."""
    panel = viz.render_provenance_panel(claim_id)
    return widgets.VBox([panel])

def show(*items):
    """Display data frames, HTML, or figures, one per argument."""
    for item in items:
        display(item)

def confidence_legend():
    """Show the colour coding applied to confidence across all figures."""
    display(HTML(
        "<b>Confidence colour coding:</b> "
        f"<span style='color:{viz.CONFIDENCE_COLORS['high']}'>high</span> &middot; "
        f"<span style='color:{viz.CONFIDENCE_COLORS['medium']}'>medium</span> &middot; "
        f"<span style='color:{viz.CONFIDENCE_COLORS['low']}'>low</span>"
    ))

## Provenance

Each claim below is rendered with its evidence class, confidence, falsifier status, cited sources and matched extraction-log entries. The colour of the confidence label is the same colour used on the chart points below.

In [ ]:
for claim_id in CLAIM_IDS:
    display(HTML(f"<h4>{claim_id} &mdash; {claims.set_index('local_id').loc[claim_id, 'claim_text']}</h4>"))
    display(provenance(claim_id))
confidence_legend()

## Primary evidence

### Which specification versions does the corpus actually record?

The corpus records Matter versions as free text in a single metric cell rather than as a dated release table. The timeline below is therefore drawn only from what is in the database: the version list in `M002`, the current major version in `M001`, and the claims that state what a specific version added (`C003` for 1.4, `C009` for the 1.5 camera support, `C011` for 1.6). There are no release dates in the corpus for 1.4 or 1.6, so the x-axis is version ordinal, not calendar time, and no dates are invented.

In [ ]:
# Which versions does the corpus actually record? M002 holds the list; M001
# holds the current major version. Both are authored strings, so they are read
# and parsed rather than restated.
_versions = viz.section_metrics("Documented developments")
_versions = _versions[_versions["local_id"] == "M002"]
_current = viz.section_metrics("Device Types & Feature Consistency")
_current = _current[_current["local_id"] == "M001"]

recorded = _versions["value"].iloc[0] if not _versions.empty else "(not recorded)"
current = _current["value"].iloc[0] if not _current.empty else "(not recorded)"

# The capability attached to each version comes from the claim text of the
# listed claim, not from a capability table the corpus does not have.
VERSION_CLAIMS = [
    ("1.4", "C003", "Enhanced Multi-Admin; certifiable routers; energy devices"),
    ("1.5", "C009", "Camera support on Echo devices (March 2026)"),
    ("1.6", "C011", "NFC commissioning; Joint Fabric"),
]
RECORDED = """ + repr("1.4, 1.4.2, 1.5, 1.6") + r"""
points = [
    {
        "version": version,
        "ordinal": index,
        "capability": capability,
        "claim": claim,
        "claim_text": claims.set_index("local_id").loc[claim, "claim_text"],
        "in_recorded_list": version in RECORDED,
    }
    for index, (version, claim, capability) in enumerate(VERSION_CLAIMS)
]
timeline = pd.DataFrame(points)
show(timeline[["version", "capability", "claim", "in_recorded_list"]])

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=list(timeline["ordinal"]),
    y=[1] * len(timeline),
    mode="markers+text",
    marker=dict(
        size=[18 if seen else 10 for seen in timeline["in_recorded_list"]],
        color=[viz.CONFIDENCE_COLORS["high"] if seen
               else viz.CONFIDENCE_COLORS[None]
               for seen in timeline["in_recorded_list"]],
        line=dict(width=1, color="#333333"),
    ),
    text=list(timeline["version"]),
    textposition="top center",
    customdata=list(zip(timeline["capability"], timeline["claim"],
                        timeline["claim_text"], timeline["in_recorded_list"])),
    hovertemplate=("Matter %{text}<br>recorded capability: %{customdata[0]}"
                   "<br>claim %{customdata[1]}: %{customdata[2]}"
                   "<br>in M002 version list: %{customdata[3]}<extra></extra>"),
))
_x = list(timeline["ordinal"])
for lo, hi in zip(_x, _x[1:]):
    fig.add_shape(type="line", x0=lo, x1=hi, y0=1, y1=1,
                  line=dict(color="#999999", width=1, dash="dot"))
fig.update_layout(
    title="Matter versions in the corpus (ordinal axis; the corpus records no dates)",
    xaxis=dict(title="Version ordinal", tickvals=_x,
               ticktext=list(timeline["version"]), range=[-0.5, len(_x) - 0.5]),
    yaxis=dict(visible=False, range=[0.5, 1.5]),
    height=340,
    showlegend=False,
)
fig.show()

display(HTML(
    f"<p>Recorded version list (<code>M002</code>): <b>{recorded}</b>"
    f" &mdash; current major version at the research date "
    f"(<code>M001</code>): <b>{current}</b>. Both are stored as authored "
    "strings, which is why the parser in <code>viz_core.parse_numbers</code> "
    "is unit-tested separately. A hollow, small marker above means the version "
    "is represented by a claim but not by the version-list metric.</p>"
))

## Supporting context

### What the claims say the standard buys, and what it does not

`C001` and `C002` are the documented factual base. `C005` and `C006` are the synthesis claims: standardisation reduces friction but commoditises basic features, and certification versions plus optional clusters plus vendor extensions plus uneven ecosystem support can still produce fragmentation. `C006` is the only claim in this cluster with a falsifier written from the source, and it is the reason the cluster is worth plotting at all: the fragmentation reading is falsifiable, while the core-idea claims are not.

### Claim and source inventory

`C001` and `C002` carry no falsifier (see the gaps section); `C005` and `C006` do. The distinction is visible in the `no_falsifier` column below.

The Stage 2 schema carries no foreign key from claims to metrics, entities or sources. `viz_core` derives those links by shared section and shared source id, and labels the reason in a `linkage` column, so the inference behind every number on screen stays inspectable.

In [ ]:
display(HTML(
    "<h4>Claim cluster evidence table</h4>"
    "<p>Every claim in this cluster, with its evidence class, confidence, "
    "workflow stage and cited source ids. This is the table the figures above "
    "are built from.</p>"
))
show(
    claims[["local_id", "claim_type", "evidence_marker", "confidence",
            "workflow_stage", "source_refs", "falsifier_missing", "section"]]
    .rename(columns={
        "local_id": "claim", "claim_type": "type", "evidence_marker": "ev",
        "confidence": "conf", "workflow_stage": "stage",
        "source_refs": "sources", "falsifier_missing": "no_falsifier",
    })
)

display(HTML("<h4>Sources cited by the cluster</h4>"))
show(sources[["local_id", "title", "publisher", "classification",
              "publication_date", "url"]].rename(
    columns={"local_id": "source"}))

## Uncertainty and gaps

### What this notebook cannot show

- **No release dates.** The corpus records Matter 1.4 and 1.6 as version identifiers with no dates; the one date present (`M005`, Amazon Echo 1.5 support, March 2026) is an ecosystem release, not a specification release. The x-axis is ordinal for that reason.
- **No per-ecosystem device-type counts.** `M004` records 58 device types for Samsung SmartThings only; the corpus has no equivalent count for Apple, Google or Amazon, so an ecosystem parity matrix would be partly invented. That notebook is deferred until parity data exists.
- **Capability attributions are claim-scoped.** Each capability label on the timeline comes from the claim text of `C003`, `C009` and `C011`, not from an independent capability table.

In [ ]:
display(HTML("<h4>Claims in this cluster with no falsifier</h4>"
             "<p>A missing falsifier means the corpus recorded no observation "
             "that would disconfirm the claim. The panel above flags these "
             "claims in <code>no_falsifier</code>.</p>"))
show(claims.loc[claims["falsifier_missing"], ["local_id", "claim_type",
                                              "confidence"]]
     .rename(columns={"local_id": "claim", "claim_type": "type",
                      "confidence": "conf"}))

display(HTML("<h4>Metrics with conditions not stated in the source</h4>"))
_gaps = cluster_metrics.loc[~cluster_metrics["conditions_stated"].fillna(False)]
if _gaps.empty:
    display(HTML("<p>No unstated conditions among the cluster's metrics.</p>"))
else:
    show(_gaps[["local_id", "metric_name", "value", "unit"]].rename(
        columns={"local_id": "metric", "metric_name": "name"}))

display(HTML("<h4>Extraction decisions that changed or qualified this material</h4>"))
_decisions = pd.DataFrame(
    viz.connect_db().execute(
        "SELECT local_id, decision_type, description, resolution "
        "FROM extraction_decision "
        "WHERE decision_type IN ('evidence_downgrade', 'classification_conflict', "
        "                           'falsifier_absent', 'condition_absent', "
        "                           'boundary_absent', 'scope_boundary', "
        "                           'omission') "
        "ORDER BY local_id"
    ).fetchall(),
    columns=["decision", "type", "description", "resolution"],
)
show(_decisions)

## Reproducibility

Figures are built from `viz.section_metrics` (exact `section` match) and `viz.fetch_claims` (explicit local ids). Both issue read-only SQL against `data/smarthome.duckdb`; no other file is read. The notebook is executed headless by `src/tests/test_viz_notebooks.py`, so this page is known to run.

In [ ]:
# The exact queries behind every figure above, re-run verbatim.
import duckdb
con = duckdb.connect(str(ROOT / 'data' / 'smarthome.duckdb'), read_only=True)
print('database:', con.execute('SELECT current_database()').fetchone()[0])
for table in ('stage2_claim', 'metric', 'entity', 'source_mirror',
               'predicate', 'workflow_stage', 'extraction_decision'):
    total = con.execute(f'SELECT COUNT(*) FROM {table}').fetchone()[0]
    print(f'{table:22} {total:>6} rows')
con.close()